# 🎛️ Strata Studio

<a href="https://colab.research.google.com/github/PingPhantom/Colab/blob/main/nix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

จัดการโมเดล GGUF (10 ตัว + draft/MTP/N-gram) บน **llama.cpp** พร้อม **StrataRAG** และ UI 4 แท็บ
สำหรับ Google Colab GPU (T4 ฟรีก็ใช้ได้)

**วิธีใช้:** รันเซลล์ 1 (ติดตั้งสภาพแวดล้อม) → เซลล์ 2 (เปิด UI) แล้วใช้ผ่านแท็บได้เลย

> โค้ดจริงอยู่ในแพ็กเกจ [`strata/`](./strata) — notebook เหลือแค่เซลล์ setup สั้น ๆ


In [ ]:
# @title 1) เตรียมโค้ด + ติดตั้งสภาพแวดล้อม (Nix + llama.cpp + uv) { display-mode: "form" }
# @markdown เปิด notebook เดี่ยว ๆ ใน Colab ได้เลย — เซลล์นี้จะ clone รีโปให้ถ้ายังไม่มีแพ็กเกจ `strata/`
REPO_SLUG = "PingPhantom/Colab"  # @param {type:"string"}
REPO_BRANCH = ""  # @param {type:"string"}
# @markdown เว้น `REPO_BRANCH` ว่าง = ใช้ branch หลัก (ใส่ชื่อ branch เช่น `arena/e076c6f4-colab` เพื่อทดสอบ PR)

import os
import shutil
import subprocess
import sys
import time
from pathlib import Path


def _ensure_repo(slug: str = "", branch: str = "") -> Path:
    """หาโฟลเดอร์ที่มีแพ็กเกจ `strata/` — ถ้าไม่มีให้ clone จาก GitHub (ใช้ได้ทั้ง Colab และเครื่องตัวเอง)"""
    here = Path.cwd()
    for base in [here, *here.parents]:
        if (base / "strata" / "__init__.py").is_file():
            return base
    target = Path("/content/Colab") if Path("/content").is_dir() else here / "Colab"
    if not (target / "strata" / "__init__.py").is_file():
        if target.exists():
            shutil.rmtree(target, ignore_errors=True)  # clone ค้าง/เก่า -> เริ่มใหม่
        cmd = ["git", "clone", "--depth", "1"]
        if branch:
            cmd += ["--branch", branch]
        cmd += [f"https://github.com/{slug}.git", str(target)]
        print(f"⏳ ไม่พบแพ็กเกจ strata/ — กำลัง clone {slug}{'@' + branch if branch else ''} ...")
        subprocess.run(cmd, check=True, env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    if not (target / "strata" / "__init__.py").is_file():
        raise RuntimeError(
            f"clone {slug} แล้วแต่ไม่พบแพ็กเกจ strata/ — ถ้าทดสอบ branch/PR อยู่ ให้ใส่ชื่อ branch ในช่อง REPO_BRANCH "
            "(เช่น arena/e076c6f4-colab) หรือ merge PR เข้า branch หลักก่อน"
        )
    return target


REPO_ROOT = _ensure_repo(REPO_SLUG, REPO_BRANCH)
os.chdir(REPO_ROOT)  # flake.nix / pyproject.toml / .venv จะอยู่ที่รากรีโป
sys.path.insert(0, str(REPO_ROOT))
print(f"📂 ใช้โค้ดจาก: {REPO_ROOT}")

from strata.bootstrap import ensure_environment

t0 = time.perf_counter()
env = ensure_environment()
print(f"✅ ใช้เวลาเตรียม env {time.perf_counter() - t0:.1f}s • nixpkgs {env['nixpkgs'][:7]}")


In [ ]:
# @title 2) เปิด Studio UI (พอร์ต 18080) { display-mode: "form" }
import os
import sys
from pathlib import Path

for _base in [Path.cwd(), *Path.cwd().parents, Path("/content/Colab")]:
    if (_base / "strata" / "__init__.py").is_file():
        sys.path.insert(0, str(_base))
        os.chdir(_base)
        break
else:
    raise RuntimeError("ยังไม่พบแพ็กเกจ strata/ — รันเซลล์ 1 ก่อนครับ")

from strata.notebook import launch, register_magics

register_magics()  # เปิดใช้ %%ruff ในเซลล์ใหม่ (ทางเลือก)
ui = launch()


## ใช้งานผ่านโค้ด (ไม่ต้องเปิด UI)

```python
from strata.download import install_model, remove_model
from strata.server import start_strata_server, stop_strata_server, get_openai_client

install_model("qwen35-9b-defiant-fable", auto_start_server=True)   # โหลด + สตาร์ท
client = get_openai_client()                                       # OpenAI-compatible ที่ :18080
print(client.chat.completions.create(
    model="qwen35-9b-defiant-fable",
    messages=[{"role": "user", "content": "สวัสดี"}],
).choices[0].message.content)
```

เรียก API จาก notebook/แอปอื่นได้ที่ `http://127.0.0.1:18080/v1` (OpenAI SDK, `curl`, LangChain ฯลฯ)

## RAG

```python
from strata.rag import GLOBAL_RAG

GLOBAL_RAG.add_files("/content/doc.pdf", chunk_size=500)
GLOBAL_RAG.save()          # /content/rag_store.npz (ชี้ไป Drive ได้)
print(GLOBAL_RAG.search("คำถาม", top_k=3))
```

## ตั้งค่าผ่าน environment variable

| ตัวแปร | ค่าเริ่มต้น | ใช้ทำอะไร |
| --- | --- | --- |
| `STRATA_PORT` | `18080` | พอร์ตของ API server/UI |
| `STRATA_HOME` | `/content` | โฟลเดอร์เก็บ log/คลัง RAG |
| `STRATA_MODEL_DIR` | `$STRATA_HOME/models` | ที่เก็บโมเดล GGUF |
| `STRATA_RAG_STORE` | `$STRATA_HOME/rag_store.npz` | ไฟล์คลัง RAG |
| `LLAMA_CPP_TAG` | release ล่าสุด | ปักหมุดเวอร์ชัน llama.cpp |
| `NIX_INSTALL_SHA256` | – | บังคับ checksum ของสคริปต์ติดตั้ง Nix |

รายละเอียดทั้งหมด (ข้อกำหนดเครื่อง, เทสต์, โครงสร้างโค้ด) อยู่ใน [`README.md`](./README.md)
